In [1]:
%run ./00_setup.ipynb
units = pd.read_parquet(INTERIM_DIR / "units.parquet")
leases = pd.read_parquet(INTERIM_DIR / "leases.parquet")
concessions = pd.read_parquet(INTERIM_DIR / "concessions.parquet")
asking = pd.read_parquet(INTERIM_DIR / "asking.parquet")

listings      1,061 rows  29 columns
leases        4,302 rows  35 columns
concessions   3,560 rows  18 columns
asking        3,857 rows  14 columns
Columns dropped from leases: ['available_date', 'property_type', 'furnishing', 'smoking', 'cats_allowed', 'dogs_allowed']
leases: 4,302 rows, 33 columns


## 2. Understanding the data

### 2.1 Profile of the four tables
First look: rows, columns, missing values, duplicates, date ranges. Nothing gets modified here.

In [2]:
table_profile = pd.DataFrame({
    "rows": [len(units), len(leases), len(concessions), len(asking)],
    "columns": [units.shape[1], leases.shape[1], concessions.shape[1], asking.shape[1]],
    "missing_values": [int(t.isna().sum().sum()) for t in (units, leases, concessions, asking)],
    "duplicated_rows": [int(t.duplicated().sum()) for t in (units, leases, concessions, asking)],
    "start": ["-", leases.lease_start.min().date(), concessions.concession_start.min().date(), asking.asking_date.min().date()],
    "end": ["-", leases.lease_start.max().date(), concessions.concession_start.max().date(), asking.asking_date.max().date()],
}, index=["units", "leases", "concessions", "asking"])
print("Duplicates on (unit_id, lease_start):", leases.duplicated(["unit_id", "lease_start"]).sum())
table_profile

Duplicates on (unit_id, lease_start): 0


,rows,columns,missing_values,duplicated_rows,start,end
units,1061,30,0,0,-,-
leases,4302,33,0,0,2017-01-01,2025-12-31
concessions,3560,19,0,0,2017-01-01,2027-03-01
asking,3857,17,0,0,2016-10-01,2025-12-01


### 2.2 Buildings, provinces and lease volume
We expect 6 buildings and 9 property codes, with The Met (Ottawa) as the only one in Ontario. Notice how far apart the first-lease dates are: that's our first hint of the mix effect.

In [3]:
buildings_overview = (leases.groupby(["province", "city", "building"])
                      .agg(prop_codes=("prop_code", lambda s: ", ".join(sorted(s.unique()))),
                           units=("unit_id", "nunique"), leases=("unit_id", "size"),
                           first_lease_year=("lease_year", "min")))
buildings_overview

prop_codes  units  leases  first_lease_year
province city          building                                                               
ON       Ottawa        The Met                       metcalfe    124     352              2023
QC       Laval         Daniel-Johnson                dj1, dj2    134     682              2019
                       Levesque                      levesque     84     641              2018
                       Saint-Elzear    stelz1, stelz2, stelz3    361    1790              2017
         Mont-Royal    Le Carlyle                     carlyle    192     535              2023
         Pointe-Claire Westpark                          wsp1    166     302              2024

### 2.3 The unit key: `sPropCode + sUnitCode`, never `sSite + sUnitCode`
The brief warns about this one, and it matters: pair two leases from different apartments and every same-apartment
growth number downstream is wrong. So we count distinct units under each candidate key and check they line up 1:1 with `hUnit` and `sLink`.

In [4]:
key_check = pd.Series({
    "units (unit_id = hUnit)": units.unit_id.nunique(),
    "prop_code + unit_code": units.groupby(UNIT_KEY).ngroups,
    "unit_link (sLink)": units.unit_link.nunique(),
    "site_slug + unit_code (WRONG key)": units.groupby(["site_slug", "unit_code"]).ngroups,
    "units colliding on site_slug + unit_code": int(units.duplicated(["site_slug", "unit_code"], keep=False).sum()),
})
one_to_one = (leases.groupby("unit_id")[UNIT_KEY].nunique().max().max() == 1
              and leases.groupby(UNIT_KEY).unit_id.nunique().max() == 1)
print("unit_id <-> prop_code + unit_code is 1:1:", one_to_one)
key_check.to_frame("count")

unit_id <-> prop_code + unit_code is 1:1: True


,count
units (unit_id = hUnit),1061
prop_code + unit_code,1061
unit_link (sLink),1061
site_slug + unit_code (WRONG key),918
units colliding on site_slug + unit_code,248


**Result:** `prop_code + unit_code`, `hUnit` and `sLink` all identify the 1,061 apartments; `sSite + sUnitCode` finds
fewer, because several property codes share one site with overlapping unit numbers.
We use `UNIT_KEY = ["prop_code", "unit_code"]` everywhere.

### 2.4 What do the lease fields really contain?
A few field descriptions are vague, so we check what's really in them: `term_months`, `lease_seq`, `is_renewal`, `rent_effective`.

In [5]:
term_distribution = leases.term_months.value_counts().sort_index()
first_lease_renewals = int(((leases.lease_seq == 1) & (leases.is_renewal == 1)).sum())
effective_above_contract = int((leases.rent_effective > leases.rent_contract + NUMERIC_TOLERANCE).sum())
ratio_when_no_concession = leases.loc[leases.has_concession == 0, "concession_depth"].abs().max()
depth_when_concession = leases.loc[leases.has_concession == 1, "concession_depth"].describe()

print("Lease lengths (months):", term_distribution.to_dict())
print("First leases flagged as renewals:", first_lease_renewals)
print("Leases with effective rent > contract rent:", effective_above_contract)
print(f"Max effective/contract gap when has_concession = 0: {ratio_when_no_concession:.1e}")
print("Concession depth when has_concession = 1:")
print(f"(n = {int(depth_when_concession['count']):,} leases)")
print((depth_when_concession.drop("count") * PCT).round(2).to_string())
pd.crosstab(leases.term_label, leases.term_months)

Lease lengths (months): {5.9: 76, 12.0: 4067, 17.9: 93, 23.9: 66}
First leases flagged as renewals: 0
Leases with effective rent > contract rent: 0
Max effective/contract gap when has_concession = 0: 0.0e+00
Concession depth when has_concession = 1:
(n = 2,719 leases)
mean     7.51
std      2.64
min      2.37
25%      5.43
50%      7.23
75%      9.41
max     13.92


term_months,5.9,12.0,17.9,23.9
term_label,,,,
6 Months,76,100,0,0
Long Term,0,2578,93,66
Negotiable,0,1287,0,0
Short Term,0,102,0,0


What we found:
- `term_months` is 12 for about 95 % of leases; a few run ~6, ~18 and ~24 months. `term_label` (`sLeaseTerm`) is a
  marketing label that doesn't match the real length, so we don't use it as a duration.
- A first lease (`lease_seq = 1`) is never a renewal: there, `is_renewal = 0` means "first lease", not "turnover".
  This doesn't affect us, because a first lease has no previous lease and so never forms a pair.
- `rent_effective ≤ rent_contract` always, and equal when there is no concession: `rent_effective` really is the average
  monthly rent *net* of concessions over the lease.

### 2.5 Concession codes
Per concession code: median amount, median duration, and the amount relative to the lease rent.

In [6]:
lease_rent_by_unit_year = (leases.assign(year=leases.lease_year)
                           .groupby(["unit_id", "year"]).rent_contract.first())
concessions_with_rent = concessions.assign(year=concessions.concession_start.dt.year).join(
    lease_rent_by_unit_year, on=["unit_id", "year"])
concession_codes = (concessions_with_rent.groupby("concession_code")
                    .agg(lines=("concession_amount", "size"),
                         median_amount=("concession_amount", "median"),
                         median_months=("concession_months", "median"),
                         median_amount_to_rent=("concession_amount",
                                                lambda s: (-s / concessions_with_rent.loc[s.index, "rent_contract"]).median()))
                    .sort_values("lines", ascending=False))
concession_codes.round(2)

,lines,median_amount,median_months,median_amount_to_rent
concession_code,,,,
one_time_promo_credit,1494,-1356.60,1.0,0.69
freepark,1006,-150.50,12.0,0.07
Freeothe,570,-153.49,12.0,0.07
freelock,175,-139.35,12.0,0.07
freerent,166,-129.30,12.0,0.07
Indem,69,-149.26,12.0,0.06
freeappl,44,-143.58,12.0,0.08
Referenc,36,-148.26,12.0,0.08


### 2.6 `PromoPay` is not a monthly charge
The numbers give it away: `one_time_promo_credit` (formerly `PromoPay`) is recorded over **1 month** (median) for a median amount of about **0.7 month
of rent** (column `median_amount_to_rent` above; some exceed a full month). If it really were monthly, tenants would be living almost for free,
which `rent_effective` contradicts (average depth ≈ 7.5 % when there is a concession). **Decision:** it is a one-off credit, to be spread over
`term_months` when we want it per month. The other codes (parking, locker, appliances, "other")
are worth ≈ $130–155 a month over ≈ 9–12 months. `Indem` (indemnity paid to the resident) and `Referenc` (referral credit)
are not price discounts: we report them but leave them out of the "price" discussion.

### 2.7 Does the concessions file exactly explain `rent_effective`?
We attach each concession line to the lease of the same `unit_id` whose period contains `concession_start`.
What we expect: a lease can have several concessions (n:1); a concession must fall in only one lease.

In [7]:
lease_windows = leases[["unit_id", "lease_start", "lease_end", "has_concession"]].reset_index(names="lease_row")
concession_lease_candidates = concessions.reset_index(names="concession_row").merge(lease_windows, on="unit_id", how="left")
in_window = concession_lease_candidates.concession_start.between(
    concession_lease_candidates.lease_start, concession_lease_candidates.lease_end)
concession_to_lease = concession_lease_candidates[in_window]
assert not concession_to_lease.concession_row.duplicated().any(), "a concession falls in two leases"

leases_with_concession_rows = set(concession_to_lease.lease_row)
flag_vs_rows = pd.Series({
    "concession lines": len(concessions),
    "lines attached to a lease": concession_to_lease.concession_row.nunique(),
    "lines outside any lease period": len(concessions) - concession_to_lease.concession_row.nunique(),
    "leases with has_concession=1 but no line": int(((leases.has_concession == 1)
                                                     & ~leases.index.isin(leases_with_concession_rows)).sum()),
    "leases with has_concession=0 but some lines": int(((leases.has_concession == 0)
                                                       & leases.index.isin(leases_with_concession_rows)).sum()),
})
flag_vs_rows.to_frame("count")

,count
concession lines,3560
lines attached to a lease,3356
lines outside any lease period,204
leases with has_concession=1 but no line,497
leases with has_concession=0 but some lines,264


**Decision:** the concessions file and the `has_concession` flag don't fully agree (flagged leases without
lines, lines without a lease). We can't rebuild `rent_effective` exactly. **`rent_effective` is our source of truth**
for the amount; the concessions file tells us *what kind* of discount was given (§5).

### 2.8 Checking the joins between tables
Quick sanity check: every unit in the leases, concessions and asking tables exists in `units` (join on `unit_id`, n:1 cardinality).

In [8]:
join_checks = pd.Series({
    "lease units missing from units": len(set(leases.unit_id) - set(units.unit_id)),
    "concession units missing from units": len(set(concessions.unit_id) - set(units.unit_id)),
    "asking units missing from units": len(set(asking.unit_id) - set(units.unit_id)),
    "units without any lease": len(set(units.unit_id) - set(leases.unit_id)),
    "units.unit_id unique (the 1 side of n:1)": units.unit_id.is_unique,
})
join_checks.to_frame("result")

,result
lease units missing from units,0
concession units missing from units,0
asking units missing from units,0
units without any lease,0
units.unit_id unique (the 1 side of n:1),True


### 2.9 Data dictionary: every column, its new name and what it really means
**What we do:** one row per column across the four files (columns shared by several files appear once). For each one:
the original Yardi name, the name we use, which files have it, what it **actually** contains (checked on the data, not
taken from the Yardi description), how we use it, and two facts computed live from the data (type, number of distinct values).
**Why:** the brief asks us to check what every column really represents and rename it where needed. The source files stay
untouched; all renaming happens in §0.4.

In [9]:
# (original Yardi name, our name, what it really contains, how we use it)
COLUMN_MEANINGS = [
    # --- keys (h = handle, used only for joins) ---
    ("hUnit", "unit_id", "internal ID of the apartment; 1:1 with prop_code + unit_code (§2.3)", "join key"),
    ("hProperty", "property_id", "internal ID of the property phase (9 phases: some buildings have 2 or 3)", "join only"),
    ("hBuilding", "building_id", "internal ID of the building (6 buildings)", "join only"),
    # --- apartment identity (s = stored value) ---
    ("sPropCode", "prop_code", "short code of the property phase, e.g. stelz1", "half of the apartment key"),
    ("sUnitCode", "unit_code", "apartment number inside the property; repeats across properties", "half of the apartment key, never alone"),
    ("sSite", "site_slug", "short site name used in web links", "display only, NEVER a key (248 units collide)"),
    ("sLink", "unit_link", "web link to the apartment's listing; one per apartment", "not used (1:1 with unit_id)"),
    # --- location ---
    ("sBuilding", "building", "building name", "labels and grouping"),
    ("sCity", "city", "city of the building", "labels"),
    ("sState", "province_name", "province name, turned into province = QC / ON", "decides which rent rules apply"),
    ("sAddr1", "address", "street address of the building", "not used"),
    ("sLatitude", "latitude", "building location", "not used"),
    ("sLongitude", "longitude", "building location", "not used"),
    # --- apartment features ---
    ("sBeds", "bedrooms", "number of bedrooms (0 = studio)", "mix analysis (§3), building x bedrooms forecast (§10.3)"),
    ("sBaths", "bathrooms", "number of bathrooms", "not used"),
    ("sSqft", "sqft", "floor area in square feet", "mix analysis (§3)"),
    ("sFloor", "floor", "floor the apartment is on", "not used"),
    ("sUnitType", "layout_code", "layout code, e.g. 1_1d = 1 bedroom, 1 bathroom + den", "not used"),
    ("sUnitSubtype", "bedroom_label", "text version of bedrooms: Studio, 1 Bed, 2 Bed, 3 Bed", "not used (same as bedrooms)"),
    ("sUnitDesc", "unit_description", "marketing description of the layout", "not used"),
    ("sPropType", "property_type", "always 'Apartment'", "dropped: one value, no information"),
    ("sFurnishing", "furnishing", "always 'Unfurnished'", "dropped: one value"),
    ("sSmoking", "smoking", "always 'Non-Smoking'", "dropped: one value"),
    ("sCats", "cats_allowed", "always True", "dropped: one value"),
    ("sDogs", "dogs_allowed", "always True", "dropped: one value"),
    # --- leases ---
    ("sRent (leases)", "rent_contract", "monthly rent written on the lease, before any discount", "contract growth, HCUC"),
    ("sRentEffective", "rent_effective", "average monthly rent actually collected over the lease, after discounts; never above rent_contract", "effective growth, our headline HEUC"),
    ("sConcession", "has_concession", "1 if the lease got at least one discount", "concession frequency (§5)"),
    ("sRenewal", "is_renewal -> lease_type", "1 = same tenant renews; 0 = new tenant OR the apartment's first lease", "renewal vs turnover (§6)"),
    ("sTermSeq", "lease_seq", "rank of the lease for this apartment (1 = first lease)", "checks only"),
    ("sTermMonths", "term_months", "real lease length in months (12 for about 95 % of leases)", "spreads the one-off promo credit (§5)"),
    ("sLeaseTerm", "term_label", "marketing label ('Long Term', '6 Months'...); does not match the real length", "not used as a duration"),
    ("sSignDate", "sign_date", "date the lease was signed", "not used"),
    ("sLeaseFrom", "lease_start", "date the lease (and its rent) starts; lease_year = its year", "pairing and yearly attribution (§4)"),
    ("sLeaseTo", "lease_end", "date the lease ends (can go past 2025)", "overlap check (§4)"),
    ("sAvailable", "available_date", "leases: identical to lease_start on 100 % of rows; listings: date the unit is free", "dropped from leases (redundant)"),
    # --- listings and asking rents ---
    ("sRent (listings)", "asking_rent_current", "rent currently advertised for the unit, not a signed rent", "not used in the forecast (§8.5)"),
    ("sAsOf", "asking_as_of", "month the current advertised rent refers to", "not used"),
    ("sMonth", "asking_month", "month the unit was advertised (YYYY-MM)", "context only"),
    ("sAskingRent", "asking_rent", "rent advertised during that month, not a signed rent", "context only (§8.5)"),
    # --- concessions ---
    ("sChargeCode", "concession_code", "type of discount; 8 codes cut to 8 characters (PromoPay renamed one_time_promo_credit)", "discount types (§2.5, §5.2)"),
    ("sChargeDesc", "concession_desc", "description of the code; says PromoPay is monthly, which the data contradicts (§2.6)", "checked, not trusted"),
    ("sAmount", "concession_amount", "dollars given back (negative number)", "discount size (§2.5)"),
    ("sDateFrom", "concession_start", "first day the discount applies", "links a discount to its lease (§2.7)"),
    ("sDateTo", "concession_end", "last day the discount applies", "not used"),
    ("sMonths", "concession_months", "number of months the discount covers (PromoPay: 1, i.e. one-off)", "PromoPay check (§2.6)"),
]
RENAMED_TABLES = {"listings": units, "leases": raw_leases.rename(columns={**SHARED_RENAME, **LEASE_RENAME}),
                  "concessions": concessions, "asking": asking}   # leases before columns were dropped (§0.5)


def column_facts(new_name):
    """Which renamed tables hold the column, its type and its number of distinct values (no raw values shown)."""
    base = new_name.split(" ->")[0]
    tables = [name for name, table in RENAMED_TABLES.items() if base in table.columns]
    if not tables:
        return "", "", None
    sample = RENAMED_TABLES[tables[0]][base]
    return ", ".join(tables), str(sample.dtype), int(sample.nunique())


data_dictionary = pd.DataFrame(COLUMN_MEANINGS, columns=["original column", "our name", "what it really contains", "how we use it"])
data_dictionary[["found in", "type", "distinct values"]] = pd.DataFrame(
    [column_facts(name) for name in data_dictionary["our name"]], index=data_dictionary.index)
renamed_everywhere = {**SHARED_RENAME, **LEASE_RENAME, **LISTING_RENAME, **CONCESSION_RENAME, **ASKING_RENAME}
print(f"{len(data_dictionary)} distinct columns documented; every raw column renamed:",
      all(c in renamed_everywhere for raw in (raw_listings, raw_leases, raw_concessions, raw_asking) for c in raw.columns))
with pd.option_context("display.max_rows", None, "display.max_colwidth", 90):
    display(data_dictionary)

46 distinct columns documented; every raw column renamed: True


,original column,our name,what it really contains,how we use it,found in,type,distinct values
0,hUnit,unit_id,internal ID of the apartment; 1:1 with prop_code + unit_code (§2.3),join key,"listings, leases, concessions, asking",int64,1061
1,hProperty,property_id,internal ID of the property phase (9 phases: some buildings have 2 or 3),join only,"listings, leases, concessions, asking",int64,9
2,hBuilding,building_id,internal ID of the building (6 buildings),join only,"listings, leases, concessions, asking",int64,6
3,sPropCode,prop_code,"short code of the property phase, e.g. stelz1",half of the apartment key,"listings, leases, concessions, asking",str,9
4,sUnitCode,unit_code,apartment number inside the property; repeats across properties,"half of the apartment key, never alone","listings, leases, concessions, asking",str,519
5,sSite,site_slug,short site name used in web links,"display only, NEVER a key (248 units collide)","listings, leases, concessions, asking",str,6
6,sLink,unit_link,web link to the apartment's listing; one per apartment,not used (1:1 with unit_id),"listings, leases",str,1061
7,sBuilding,building,building name,labels and grouping,"listings, leases, concessions, asking",str,6
8,sCity,city,city of the building,labels,"listings, leases, concessions, asking",str,4
9,sState,province_name,"province name, turned into province = QC / ON",decides which rent rules apply,"listings, leases, concessions, asking",str,2


In [10]:
data_dictionary.to_parquet(INTERIM_DIR / "data_dictionary.parquet")